# Ariane - All 932 macros: two-stage placement + LLM-guided hard ordering

Places **all 932 ariane macros** (133 hard SRAMs + 799 soft `Grp_*` clusters), **text-only**.

**Two stages:** (1) the proven wiremask greedy places the 133 **hard** macros in an order we
search/LLM-guide; (2) the 799 **soft** clusters are filled by a deterministic coordinate-median
sweep with the hard macros fixed. `comp_res` over all 932 gives the full-932 HPWL.

Only the **hard order** is searched -- the soft stage is identical & deterministic for every
method, so methods differ only in the hard order. The full-932 HPWL (~6e5) is a **new baseline**,
NOT comparable to the old hard-only ~8.4e4.

| method | what the LLM does | cost |
|---|---|---|
| strong baseline | none (topology order) | free |
| random control | none (random block moves) | free |
| LLM full-order | proposes order of all 133 hard | paid, text |
| LLM hub-prefix | proposes ~50 hub macros to place first | paid, text |

Judge LLM *value* by whether the LLM rows beat the **random control**, not just the baseline.

## Cell 1 - Setup (clone repo, gym fallback, protobuf, anthropic)
Needs `strong_search.py` and `two_stage.py` pushed to the repo.

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone","--depth","1",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("pulled latest")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    # local fallback only -- on Kaggle real gym is present. two_stage uses
    # run_greedy/PlaceEnv directly (no gym.make), so a stub is enough.
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    reg=types.ModuleType("gym.envs.registration"); envs=types.ModuleType("gym.envs")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    def _register(*a,**k): pass
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    reg.register=_register; gym.envs=envs
    sys.modules["gym"]=gym; sys.modules["gym.spaces"]=spaces
    sys.modules["gym.envs"]=envs; sys.modules["gym.envs.registration"]=reg
    print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

need=["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
      "region_constraint.py","parse_netlist.py","strong_search.py","two_stage.py",
      "matrix_eval.py","ariane/netlist.pb.txt"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push to repo): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 - Sanity check (932 = 133 hard + 799 soft)

In [ ]:
from place_db import PlaceDB
placedb = PlaceDB("ariane")
hard = [n for n in placedb.node_info if placedb.node_info[n].get("is_hard")]
soft = [n for n in placedb.node_info if not placedb.node_info[n].get("is_hard")]
print("Nodes", len(placedb.node_info), "| hard", len(hard), "| soft", len(soft),
      "| Nets", len(placedb.net_info), "| Ports", len(placedb.port_info),
      "| Canvas", placedb.max_height)
assert len(placedb.node_info) == 932 and len(hard) == 133 and len(soft) == 799
assert placedb.max_height == 357
print("sanity OK")

## Cell 3 - API key (needed for the paid LLM cells below)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

## Cell 3b - Config + strong hard order (run once)

In [ ]:
import importlib, strong_search, two_stage
importlib.reload(strong_search); importlib.reload(two_stage)
from strong_search import hard_macro_names, topology_order
import two_stage as ts

MODEL      = "claude-sonnet-4-6"   # bump to "claude-opus-4-8" for a stronger (pricier) run
MAX_ITERS  = 8                     # LLM iterations per paid cell
PATIENCE   = 3                     # early-stop after this many non-improving iters
GRID       = 224
SOFT_ITERS = 8                     # soft-fill median sweeps (deterministic)
N_RANDOM   = 12                    # no-LLM random-order control budget
PREFIX_K   = 50                    # ~ids the prefix variant asks the LLM to prioritize

# Strong (deterministic) hard order -- the floor every method is measured against.
# topology_order uses a stable md5 tiebreak, so the baseline is reproducible.
HARD = hard_macro_names(placedb)
STRONG_ORDER = topology_order(placedb, HARD)
print("hard macros:", len(STRONG_ORDER), "| MODEL:", MODEL)

## Cell 4 - Baselines: two-stage strong + random control (FREE, no API)

In [ ]:
# FREE (no API): two-stage strong baseline + no-LLM random-order control (full-932 HPWL).
strong_hpwl, strong_env = ts.two_stage_hpwl(placedb, STRONG_ORDER, grid=GRID, soft_iters=SOFT_ITERS)
print(f"strong baseline  full-932 HPWL = {strong_hpwl:.4e}  (placed {len(strong_env.node_pos)})")

rand = ts.random_order_control(placedb, STRONG_ORDER, grid=GRID, n_evals=N_RANDOM,
                               soft_iters=SOFT_ITERS, verbose=True)
print(f"random control   full-932 HPWL = {rand['best_hpwl']:.4e} "
      f"({100*(strong_hpwl-rand['best_hpwl'])/strong_hpwl:+.2f}% vs strong)")

rows = [
    dict(method="strong baseline",     hpwl=strong_hpwl,        improvement=0.0,
         calls=0, out_tokens=0),
    dict(method="random control",      hpwl=rand["best_hpwl"],
         improvement=100*(strong_hpwl-rand["best_hpwl"])/strong_hpwl, calls=0, out_tokens=0),
]

## Cell 5 - LLM variant 1: full hard order (PAID, text)

In [ ]:
# PAID (text only): LLM proposes a FULL order of all 133 hard macros.
res_full = ts.llm_full_order_search(placedb, STRONG_ORDER, grid=GRID, model=MODEL,
                                    max_iters=MAX_ITERS, patience=PATIENCE, soft_iters=SOFT_ITERS)
imp = 100*(strong_hpwl-res_full["best_hpwl"])/strong_hpwl
print(f"\nLLM full-order   full-932 HPWL = {res_full['best_hpwl']:.4e} ({imp:+.2f}% vs strong) "
      f"| calls={res_full['calls']} out_tokens={res_full['out_tokens']}")
row_full = dict(method="LLM full-order", hpwl=res_full["best_hpwl"], improvement=imp,
                calls=res_full["calls"], out_tokens=res_full["out_tokens"])

## Cell 6 - LLM variant 2: hub prefix (PAID, text)

In [ ]:
# PAID (text only): LLM proposes only a short PRIORITY PREFIX of hub macros;
# the rest keep the strong baseline order. Far fewer output tokens than full-order.
res_pre = ts.llm_prefix_search(placedb, STRONG_ORDER, grid=GRID, model=MODEL,
                               max_iters=MAX_ITERS, patience=PATIENCE, k=PREFIX_K,
                               soft_iters=SOFT_ITERS)
imp = 100*(strong_hpwl-res_pre["best_hpwl"])/strong_hpwl
print(f"\nLLM hub-prefix   full-932 HPWL = {res_pre['best_hpwl']:.4e} ({imp:+.2f}% vs strong) "
      f"| calls={res_pre['calls']} out_tokens={res_pre['out_tokens']}")
row_pre = dict(method="LLM hub-prefix", hpwl=res_pre["best_hpwl"], improvement=imp,
               calls=res_pre["calls"], out_tokens=res_pre["out_tokens"])

## Cell 7 - Combined results table + CSV + best layout

In [ ]:
# Combined table (run after whichever cells above you ran; missing ones are skipped).
import csv
all_rows = list(rows)
for _v in ["row_full", "row_pre"]:
    if _v in globals():
        all_rows.append(globals()[_v])

print("\n" + "="*72)
print(f"{'method':<20}{'full-932 HPWL':>16}{'vs strong':>12}{'calls':>7}{'out_tok':>10}")
print("-"*72)
for r in all_rows:
    print(f"{r['method']:<20}{r['hpwl']:>16.4e}{r['improvement']:>+11.2f}%"
          f"{r['calls']:>7}{r['out_tokens']:>10}")
print("="*72)
print("Judge LLM value by whether the LLM rows beat the RANDOM CONTROL, not just the baseline.")

with open("/kaggle/working/twostage_932.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["method","hpwl","improvement","calls","out_tokens"])
    w.writeheader(); w.writerows(all_rows)
print("saved /kaggle/working/twostage_932.csv")

# optional: best-layout figure from whichever LLM run scored best
best = min([("strong", strong_env, strong_hpwl)] +
           ([("random", rand["best_env"], rand["best_hpwl"])] if "rand" in globals() else []) +
           ([("full", res_full["best_env"], res_full["best_hpwl"])] if "res_full" in globals() else []) +
           ([("prefix", res_pre["best_env"], res_pre["best_hpwl"])] if "res_pre" in globals() else []),
           key=lambda t: t[2])
try:
    best[1].save_fig(f"/kaggle/working/best_{best[0]}_932.png")
    from IPython.display import Image, display
    print(f"best layout: {best[0]} ({best[2]:.4e})")
    display(Image(f"/kaggle/working/best_{best[0]}_932.png"))
except Exception as e:
    print("figure skipped:", e)

## Reframe: cost / quality trade-off vs MaskPlace RL (LEGAL placement)

The two-stage HPWL above is **optimistic** — `soft_fill` lets the 799 soft clusters overlap,
which buys artificially low wirelength. Below we score every method on the **legal**
(zero-overlap, in-canvas) metric and put MaskPlace's published numbers next to it.

- `proxy_vs_legal` — does minimizing the overlapping proxy even track the legal HPWL?
- `evaluate_trade_off` — legal bbox/MST wirelength, wall-clock, and \$ token cost per method,
  plus the MaskPlace RL reference row (paper arXiv 2211.13382, ariane, grid 224, 0.00% overlap).

In [ ]:
import importlib, two_stage, trade_off_eval
importlib.reload(two_stage); importlib.reload(trade_off_eval)
import trade_off_eval as te

# 1) Proxy fidelity (FREE, no API): is the overlapping HPWL a good stand-in for legal HPWL?
pf = te.proxy_vs_legal(placedb, STRONG_ORDER, grid=GRID, refine=2, n=20, verbose=True)

# 2) Trade-off table. run_llm=False -> FREE rows only; set True with an API key for LLM rows.
rows = te.evaluate_trade_off(model=MODEL, grid=GRID, refine=2, max_iters=MAX_ITERS,
                             patience=PATIENCE, soft_iters=SOFT_ITERS, n_random=N_RANDOM,
                             prefix_k=PREFIX_K, run_llm=True, verbose=True)
te.print_trade_off(rows)
te.save_csv(rows, 'trade_off.csv')

## LLM-guided LEGAL placement of all 932 macros (PAID, the deliverable)

`legal=True` makes the LLM order search optimize the **zero-overlap** layout in-loop, so
`res['best_env']` is a real legal placement of all 932 macros (directly comparable to the
MaskPlace paper). Needs an API key (Cell 3).

In [ ]:
import importlib, two_stage; importlib.reload(two_stage); import two_stage as ts

# LLM proposes the hard order; every candidate is scored on the LEGAL (zero-overlap) layout.
res = ts.llm_full_order_search(placedb, STRONG_ORDER, grid=GRID, model=MODEL,
                               max_iters=MAX_ITERS, patience=PATIENCE, soft_iters=SOFT_ITERS,
                               legal=True, refine=2)
env = res['best_env']

# verify legality of the LLM-guided 932-macro placement
rects = list(env.node_pos.values()); ov = 0
for i in range(len(rects)):
    xi,yi,sxi,syi = rects[i]
    for j in range(i+1, len(rects)):
        xj,yj,sxj,syj = rects[j]
        if xi<xj+sxj and xj<xi+sxi and yi<yj+syj and yj<yi+syi: ov += 1
g = GRID*2
oob = sum(1 for (x,y,sx,sy) in env.node_pos.values() if x<0 or y<0 or x+sx>g or y+sy>g)
print(f'LLM-guided LEGAL placement: macros={len(env.node_pos)} overlaps={ov} out_of_canvas={oob}')
print(f"legal bbox HPWL = {res['best_hpwl']:.4e}  (grid {g})  | LLM calls={res['calls']} out_tokens={res['out_tokens']}")
print('MaskPlace paper (ariane, grid 224, legal): MST wirelength = 1.463e6')